# Notebook 06: ADAPT-VQE vs UCCSD-VQE
## Adaptive vs Fixed Ansatz for Alkenes and Alkynes on NISQ Hardware

**Central question:** For molecules with strong π-correlation (alkynes), does
ADAPT-VQE's adaptive circuit growth outperform fixed UCCSD-VQE in both
accuracy *and* circuit depth — the two key metrics for NISQ viability?

**Reference:** Grimsley et al., *Nat. Commun.* **10**, 3007 (2019)

**Key fix in this version:** ADAPT-VQE inner loop uses scipy `L-BFGS-B` with
analytic gradients via PennyLane's parameter-shift rule. The gradient screening
uses the finite-difference commutator |⟨ψ|[H,Aₖ]|ψ⟩| to rank operators,
faithfully reproducing the original Grimsley et al. algorithm.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/quantum-alkene-alkyne-pyscf/blob/main/notebooks/06_adapt_vqe_comparison.ipynb)

In [ ]:
# CELL 1: Install
import sys, subprocess
pkgs = ['pyscf>=2.5','openfermion>=1.6','openfermionpyscf>=0.5',
        'pennylane>=0.38','scipy','tqdm','seaborn','pandas']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('Ready.')

In [ ]:
# CELL 2: Imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
from tqdm.auto import tqdm, trange
from scipy.optimize import minimize
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

from openfermion.chem import MolecularData
from openfermion import get_fermion_operator, jordan_wigner
from openfermion.utils import count_qubits
try:
    from openfermion.transforms import freeze_orbitals
except ImportError:
    from openfermion.utils import freeze_orbitals
from openfermionpyscf import run_pyscf
import pennylane as qml
from pennylane import qchem
print(f'PennyLane {qml.__version__} | Imports OK.')

In [ ]:
# CELL 3: PennyLane compatibility shim
_PL_NEW = tuple(int(x) for x in qml.__version__.split('.')[:2]) >= (0, 38)

def _pauli_gate(name, wire):
    return {'X': qml.PauliX, 'Y': qml.PauliY, 'Z': qml.PauliZ}[name](wire)

def of_to_pl(qubit_op):
    coeffs, ops = [], []
    for term, c in qubit_op.terms.items():
        if abs(np.real(c)) < 1e-15:
            continue
        coeffs.append(np.real(c))
        if not term:
            ops.append(qml.Identity(0))
        else:
            gates = [_pauli_gate(p, i) for i, p in term]
            ops.append(gates[0] if len(gates) == 1 else qml.prod(*gates))
    if _PL_NEW:
        return qml.ops.LinearCombination(np.array(coeffs), ops)
    return qml.Hamiltonian(np.array(coeffs), ops)

print('Hamiltonian builder ready.')

In [ ]:
# CELL 4: Build molecules and Hamiltonians
molecules = {
    'ethylene': {
        'geometry': [
            ('C',(0.000,0.000,0.000)),('C',(0.000,0.000,1.339)),
            ('H',(0.000,0.926,-0.546)),('H',(0.000,-0.926,-0.546)),
            ('H',(0.000,0.926,1.885)),('H',(0.000,-0.926,1.885)),
        ],
        'freeze_core': [0, 1, 2],
        'frozen_e': 6,
        'pi_bonds': 1,
        'color': 'steelblue',
    },
    'acetylene': {
        'geometry': [
            ('C',(0.000,0.000,0.000)),('C',(0.000,0.000,1.203)),
            ('H',(0.000,0.000,-1.063)),('H',(0.000,0.000,2.266)),
        ],
        'freeze_core': [0],
        'frozen_e': 2,
        'pi_bonds': 2,
        'color': 'darkorange',
    },
}

mol_data, hamiltonians = {}, {}

for name, info in molecules.items():
    md = MolecularData(geometry=info['geometry'], basis='sto-3g',
                       multiplicity=1, charge=0, description=name)
    md = run_pyscf(md, run_scf=True, run_ccsd=True, run_fci=True)
    mol_data[name] = md
    fham = get_fermion_operator(md.get_molecular_hamiltonian())
    afham = freeze_orbitals(fham, occupied=info['freeze_core'], virtual=[])
    jw = jordan_wigner(afham)
    nq = count_qubits(jw)
    ne = md.n_electrons - info['frozen_e']
    hamiltonians[name] = {'jw': jw, 'pl': of_to_pl(jw), 'n_qubits': nq, 'n_electrons': ne}
    corr = (md.fci_energy - md.hf_energy)*1000
    print(f'{name}: {nq} active qubits | {ne} active e⁻ | '          f'FCI={md.fci_energy:.6f} Ha | corr={corr:.2f} mHa | π-bonds={info["pi_bonds"]}')

In [ ]:
# CELL 5: UCCSD-VQE runner (fixed ansatz, Adam optimizer)
# Uses Adam instead of plain gradient descent: faster convergence,
# adaptive per-parameter learning rates, more robust in practice.

def run_uccsd_vqe(H_pl, n_qubits, n_electrons, max_iter=300,
                  lr=0.05, conv_tol=1e-9, seed=42):
    singles, doubles = qchem.excitations(n_electrons, n_qubits)
    hf_state = qchem.hf_state(n_electrons, n_qubits)
    dev = qml.device('default.qubit', wires=n_qubits)

    @qml.qnode(dev, diff_method='best')
    def circuit(params):
        qml.BasisState(hf_state, wires=range(n_qubits))
        qml.AllSinglesDoubles(params, wires=range(n_qubits),
            hf_state=hf_state, singles=singles, doubles=doubles)
        return qml.expval(H_pl)

    np.random.seed(seed)
    n_p = len(singles) + len(doubles)
    params = np.random.uniform(-0.01, 0.01, n_p, requires_grad=True)
    opt = qml.AdamOptimizer(stepsize=lr, beta1=0.9, beta2=0.999)
    energies = []
    for _ in range(max_iter):
        params, energy = opt.step_and_cost(circuit, params)
        energies.append(float(energy))
        if len(energies) > 10 and abs(energies[-1]-energies[-2]) < conv_tol:
            break

    # Realistic CNOT estimates: UCCSD doubles ~ 8 CNOTs, singles ~ 2 CNOTs
    est_cnots = len(doubles) * 8 + len(singles) * 2
    return {
        'energy': energies[-1], 'history': energies,
        'n_params': n_p, 'est_cnots': est_cnots,
    }

print('UCCSD-VQE runner defined.')

In [ ]:
# CELL 6: ADAPT-VQE runner (Grimsley et al. Nat. Commun. 2019)
#
# Algorithm:
#   1. Screen all pool operators by |gradient| = |⟨ψ|[H, Aₖ]|ψ⟩|
#      using the parameter-shift rule: grad ≈ (f(+π/2) - f(-π/2)) / 2
#   2. Append the highest-gradient operator to the ansatz
#   3. Re-optimise ALL parameters with scipy L-BFGS-B + analytic gradients
#   4. Repeat until max|grad| < threshold
#
# Key fix vs original notebook: jac= is now computed via qml.grad of the
# adapt_circuit qnode (requires_grad=True on params), not passed as a
# PennyLane gradient fn directly to scipy (which breaks the interface).

def run_adapt_vqe(H_pl, n_qubits, n_electrons,
                  grad_thresh=1e-3, max_ops=20, max_inner=500):
    singles, doubles = qchem.excitations(n_electrons, n_qubits)
    hf_state = qchem.hf_state(n_electrons, n_qubits)
    pool = [('S', s) for s in singles] + [('D', d) for d in doubles]
    dev = qml.device('default.qubit', wires=n_qubits)

    selected = []
    params = np.array([], dtype=float)
    energy_history = []

    print(f'  Pool: {len(pool)} operators | threshold={grad_thresh}')

    for adapt_step in range(max_ops):
        # ── Gradient screening (parameter-shift, one-parameter probe) ──
        def make_probe(exc, op_type):
            @qml.qnode(dev, diff_method='parameter-shift')
            def probe(theta):
                qml.BasisState(hf_state, wires=range(n_qubits))
                for idx, (t2, e2) in enumerate(selected):
                    if t2 == 'S': qml.SingleExcitation(params[idx], wires=e2)
                    else:          qml.DoubleExcitation(params[idx], wires=e2)
                if op_type == 'S': qml.SingleExcitation(theta[0], wires=exc)
                else:               qml.DoubleExcitation(theta[0], wires=exc)
                return qml.expval(H_pl)
            return probe

        grads = []
        for op_type, exc in tqdm(pool, desc=f'  ADAPT {adapt_step+1} screening', leave=False):
            probe_fn = make_probe(exc, op_type)
            theta0 = np.array([0.0], requires_grad=True)
            g = abs(float(qml.grad(probe_fn)(theta0)[0]))
            grads.append(g)

        max_g = max(grads)
        best  = int(np.argmax(grads))
        print(f'  ADAPT step {adapt_step+1:2d}: max|∇|={max_g:.5f} → '              f'{pool[best][0]}_{list(pool[best][1])}')
        if max_g < grad_thresh:
            print(f'  Converged (max|∇| = {max_g:.2e} < {grad_thresh})')
            break

        selected.append(pool[best])
        params = np.append(params, 0.0)

        # ── Inner VQE re-optimisation with L-BFGS-B + analytic gradient ──
        current_selected = list(selected)

        @qml.qnode(dev, diff_method='parameter-shift')
        def adapt_circuit(p):
            qml.BasisState(hf_state, wires=range(n_qubits))
            for idx, (op_type, exc) in enumerate(current_selected):
                if op_type == 'S': qml.SingleExcitation(p[idx], wires=exc)
                else:               qml.DoubleExcitation(p[idx], wires=exc)
            return qml.expval(H_pl)

        # Wrap so scipy receives plain floats, PennyLane gets requires_grad arrays
        def cost_fn(p_np):
            p_pl = np.array(p_np, requires_grad=False)
            return float(adapt_circuit(p_pl))

        def grad_fn(p_np):
            p_pl = np.array(p_np, requires_grad=True)
            g_pl = qml.grad(adapt_circuit)(p_pl)
            return np.array(g_pl, dtype=float)

        res = minimize(
            cost_fn, params,
            jac=grad_fn,
            method='L-BFGS-B',
            options={'maxiter': max_inner, 'ftol': 1e-14, 'gtol': 1e-8},
        )
        params = res.x
        energy_history.append(float(res.fun))
        print(f'           E = {res.fun:.8f} Ha | converged={res.success}')

    n_s = sum(1 for t, _ in selected if t == 'S')
    n_d = sum(1 for t, _ in selected if t == 'D')
    return {
        'energy': energy_history[-1] if energy_history else np.nan,
        'history': energy_history,
        'n_params': len(selected),
        'n_operators': len(selected),
        'est_cnots': n_s * 2 + n_d * 8,
        'selected': selected,
    }

print('ADAPT-VQE runner defined.')

In [ ]:
# CELL 7: Run both methods on Ethylene
name = 'ethylene'
H    = hamiltonians[name]['pl']
nq   = hamiltonians[name]['n_qubits']
ne   = hamiltonians[name]['n_electrons']
fci  = mol_data[name].fci_energy

print('=== Ethylene (C=C, 1 π bond) ===')
print('--- UCCSD-VQE ---')
uccsd_eth = run_uccsd_vqe(H, nq, ne)
print(f'UCCSD: E={uccsd_eth["energy"]:.8f} Ha | '      f'Err={abs(uccsd_eth["energy"]-fci)*1e3:.3f} mHa | '      f'Params={uccsd_eth["n_params"]} | CNOTs≈{uccsd_eth["est_cnots"]}')

print('--- ADAPT-VQE ---')
adapt_eth = run_adapt_vqe(H, nq, ne)
print(f'ADAPT: E={adapt_eth["energy"]:.8f} Ha | '      f'Err={abs(adapt_eth["energy"]-fci)*1e3:.3f} mHa | '      f'Ops={adapt_eth["n_operators"]} | CNOTs≈{adapt_eth["est_cnots"]}')
print(f'ADAPT selected operators: {adapt_eth["selected"]}')

for label, res in [('UCCSD', uccsd_eth), ('ADAPT', adapt_eth)]:
    err = abs(res['energy'] - fci) * 1000
    print(f'{label}: {"✓ chem. acc." if err < 1.6 else "✗ above chem. acc."} ({err:.3f} mHa)')

In [ ]:
# CELL 8: Run both methods on Acetylene
name = 'acetylene'
H    = hamiltonians[name]['pl']
nq   = hamiltonians[name]['n_qubits']
ne   = hamiltonians[name]['n_electrons']
fci  = mol_data[name].fci_energy

print('=== Acetylene (C≡C, 2 π bonds) ===')
print('--- UCCSD-VQE ---')
uccsd_acc = run_uccsd_vqe(H, nq, ne)
print(f'UCCSD: E={uccsd_acc["energy"]:.8f} Ha | '      f'Err={abs(uccsd_acc["energy"]-fci)*1e3:.3f} mHa | '      f'Params={uccsd_acc["n_params"]} | CNOTs≈{uccsd_acc["est_cnots"]}')

print('--- ADAPT-VQE ---')
adapt_acc = run_adapt_vqe(H, nq, ne)
print(f'ADAPT: E={adapt_acc["energy"]:.8f} Ha | '      f'Err={abs(adapt_acc["energy"]-fci)*1e3:.3f} mHa | '      f'Ops={adapt_acc["n_operators"]} | CNOTs≈{adapt_acc["est_cnots"]}')

for label, res in [('UCCSD', uccsd_acc), ('ADAPT', adapt_acc)]:
    err = abs(res['energy'] - fci) * 1000
    print(f'{label}: {"✓ chem. acc." if err < 1.6 else "✗ above chem. acc."} ({err:.3f} mHa)')

In [ ]:
# CELL 9: Publication-ready benchmark plots (3-panel)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Panel 1: Energy convergence ───────────────────────────────────────
ax = axes[0]
ax.plot(uccsd_eth['history'], 'b-', lw=2, label='UCCSD (ethylene)')
ax.plot(range(len(adapt_eth['history'])), adapt_eth['history'],
        'b--o', ms=6, lw=2, label='ADAPT (ethylene)')
ax.axhline(mol_data['ethylene'].fci_energy, color='b', ls=':', alpha=0.5,
           label='FCI (ethylene)')
ax.plot(uccsd_acc['history'], color='darkorange', lw=2, label='UCCSD (acetylene)')
ax.plot(range(len(adapt_acc['history'])), adapt_acc['history'],
        color='darkorange', ls='--', marker='o', ms=6, lw=2, label='ADAPT (acetylene)')
ax.axhline(mol_data['acetylene'].fci_energy, color='darkorange', ls=':', alpha=0.5,
           label='FCI (acetylene)')
ax.set_xlabel('Iteration', fontsize=11)
ax.set_ylabel('Energy (Ha)', fontsize=11)
ax.set_title('Energy Convergence', fontsize=12)
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

# ── Panel 2: CNOT count comparison ────────────────────────────────────
ax2 = axes[1]
mol_labels = ['Ethylene\n(C=C)', 'Acetylene\n(C≡C)']
uccsd_cnots = [uccsd_eth['est_cnots'], uccsd_acc['est_cnots']]
adapt_cnots = [adapt_eth['est_cnots'], adapt_acc['est_cnots']]
x = np.arange(2); w = 0.35
bars1 = ax2.bar(x - w/2, uccsd_cnots, w, label='UCCSD-VQE', color='steelblue', alpha=0.85)
bars2 = ax2.bar(x + w/2, adapt_cnots, w, label='ADAPT-VQE', color='darkorange', alpha=0.85)
ax2.bar_label(bars1, padding=2, fontsize=9)
ax2.bar_label(bars2, padding=2, fontsize=9)
ax2.set_xticks(x); ax2.set_xticklabels(mol_labels)
ax2.set_ylabel('Est. CNOT count', fontsize=11)
ax2.set_title('Circuit Depth (NISQ metric)', fontsize=12)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# ── Panel 3: Error vs FCI (mHa) with chemical accuracy line ──────────
ax3 = axes[2]
labels = ['Ethylene\nUCCSD', 'Ethylene\nADAPT', 'Acetylene\nUCCSD', 'Acetylene\nADAPT']
errors = [
    abs(uccsd_eth['energy'] - mol_data['ethylene'].fci_energy) * 1000,
    abs(adapt_eth['energy'] - mol_data['ethylene'].fci_energy) * 1000,
    abs(uccsd_acc['energy'] - mol_data['acetylene'].fci_energy) * 1000,
    abs(adapt_acc['energy'] - mol_data['acetylene'].fci_energy) * 1000,
]
colors = ['steelblue', 'steelblue', 'darkorange', 'darkorange']
hatches = ['', '/', '', '//']
bars = ax3.bar(labels, errors, color=colors, edgecolor='k', lw=0.7)
for bar, h in zip(bars, hatches): bar.set_hatch(h)
ax3.bar_label(bars, fmt='%.3f', padding=2, fontsize=8)
ax3.axhline(1.6, color='red', ls='--', lw=1.5, label='Chemical accuracy (1.6 mHa)')
ax3.set_ylabel('|E_VQE − E_FCI| (mHa)', fontsize=11)
ax3.set_title('Error vs FCI Reference', fontsize=12)
ax3.legend(fontsize=9)
ax3.grid(axis='y', alpha=0.3)

plt.suptitle('ADAPT-VQE vs UCCSD-VQE · Alkene & Alkyne Benchmark (STO-3G, active space)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('adapt_vs_uccsd_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as adapt_vs_uccsd_benchmark.png')

In [ ]:
# CELL 10: Publication-ready benchmark table (CSV + display)
table_data = {
    'Molecule':       ['Ethylene (C=C)', 'Ethylene (C=C)', 'Acetylene (C≡C)', 'Acetylene (C≡C)'],
    'Method':         ['UCCSD-VQE', 'ADAPT-VQE', 'UCCSD-VQE', 'ADAPT-VQE'],
    'Energy (Ha)':    [uccsd_eth['energy'], adapt_eth['energy'],
                       uccsd_acc['energy'], adapt_acc['energy']],
    'ΔE_FCI (mHa)':  [
        abs(uccsd_eth['energy'] - mol_data['ethylene'].fci_energy)  * 1000,
        abs(adapt_eth['energy'] - mol_data['ethylene'].fci_energy)  * 1000,
        abs(uccsd_acc['energy'] - mol_data['acetylene'].fci_energy) * 1000,
        abs(adapt_acc['energy'] - mol_data['acetylene'].fci_energy) * 1000,
    ],
    '# Params':       [uccsd_eth['n_params'], adapt_eth['n_params'],
                       uccsd_acc['n_params'], adapt_acc['n_params']],
    'Est. CNOTs':     [uccsd_eth['est_cnots'], adapt_eth['est_cnots'],
                       uccsd_acc['est_cnots'], adapt_acc['est_cnots']],
}
df = pd.DataFrame(table_data)
df['Energy (Ha)']   = df['Energy (Ha)'].round(8)
df['ΔE_FCI (mHa)']  = df['ΔE_FCI (mHa)'].round(4)
df['Chem. acc. ✓?'] = df['ΔE_FCI (mHa)'].apply(lambda x: '✓' if x < 1.6 else '✗')
print(df.to_string(index=False))
df.to_csv('benchmark_table.csv', index=False)
print('\nSaved to benchmark_table.csv')

## Hardware Considerations for IBM Quantum

To run these circuits on real IBM Quantum hardware (notebook 04):

1. **Qubit count:** Active-space selection keeps circuits at 8–12 qubits,
   well within 127-qubit Eagle or 433-qubit Osprey.
2. **ADAPT advantage:** Selecting only 3–8 operators vs 20+ for UCCSD gives
   shallower circuits that survive decoherence (T₂ ≈ 100 μs on Eagle).
3. **Error mitigation:** Use `ZNE` (Zero Noise Extrapolation) from
   `qiskit_ibm_runtime.options` to push accuracy toward 1.6 mHa chemical threshold.
4. **Transpilation:** Use `optimization_level=3` with Qiskit for heavy-hex
   topology. BK mapping produces more local operators → fewer SWAP gates.
5. **Shots:** 8192–16384 shots for Hamiltonian estimation;
   use Pauli grouping (commutativity partitioning) to reduce executions by ~5×.

## API Fixes Applied in This Version

| Issue | Old (broken) | Fixed |
|---|---|---|
| PennyLane Hamiltonian | `qml.Hamiltonian` | `qml.ops.LinearCombination` (PL ≥ 0.38) |
| Tensor product | `qml.operation.Tensor` | `qml.prod()` |
| Optimizer | `GradientDescentOptimizer(0.4)` | `AdamOptimizer(0.05)` |
| ADAPT gradient | `qml.grad` passed to `jac=` | Wrapper `grad_fn` with `requires_grad=True` |
| ADAPT `n_params` key | missing from return dict | Added `'n_params': len(selected)` |
| `freeze_orbitals` import | hard import from `.transforms` | try/except fallback to `.utils` |